# Tensor-hypercontraction walk resources

Compile the complete original-sized walk and inspect its native circuit costs without executing a dense full state.

Load the native compiler and reproducible inputs.

In [ ]:
from examples.synthesis.applications.chemistry.tensor_hypercontraction.overview import (
    main,
    model,
    quantum,
)
from examples.synthesis.applications.chemistry.tensor_hypercontraction.overview.layout import (
    allocate_layout,
)


Keep original precision and spin-register widths.

In [ ]:
assert (model.N_EPSILON, model.N_UD) == (4, 3)


Generate all tables, their rotation, and the sine clock.

In [ ]:
data = model.generate()
main.describe(data)


Allocate the full record, spin, and clock layout.

In [ ]:
layout = allocate_layout()
assert layout.logical_width == 83


Reserve the traversal flags for nested clock and QROM calls.

In [ ]:
assert len(layout.path) == model.N_MN + model.N_EPSILON


Compile the first PREPARE block.

In [ ]:
early = quantum.equal_superposition(data, layout)


Use native IR resource reporting.

In [ ]:
report = main.report


Inspect the resources for PREPARE part 1.

In [ ]:
report("PREPARE part 1", early)


Compile address arithmetic, alias lookup and swaps.

In [ ]:
late = quantum.finish_prepare(data, layout)


Inspect the resources for PREPARE part 2.

In [ ]:
report("PREPARE part 2", late)


Combine both PREPARE stages.

In [ ]:
prepare = quantum.empty(layout).compose(early).compose(late)


Compile SELECT, including digital rotations and phases.

In [ ]:
select = quantum.select_block(data, layout)


Inspect SELECT resources.

In [ ]:
report("SELECT", select)


Use the inverse preparation and a wide controlled phase for reflection.

In [ ]:
inverse = prepare.adjoint()


Add the original clock and spin state preparations and the inverse QFT.

In [ ]:
complete = quantum.full_walk(data, layout, prepare, inverse, select)


Materialize the full native circuit product.

In [ ]:
assert complete.to_ir().n_wires == layout.width


Report the full resources and audit the emitted reversible subcircuits.

In [ ]:
report("Full clock program", complete)
main.report_checks(data, layout)
